# Introduction
This notebook computes each patient's risk of grade 2+ parotid gland-related xerostomia using a published NTCP model [3] on a head-and-neck DICOM cohort.

The model uses the mean dose to four structures, plus whether the patient had xerostomia before treatment:
* An AI model [1] segments the parotids, masseters, mandible, submandibular glands and oral cavity.
* The Stem-Cell Rich Region (SCRR) is delineated from these with a published method [2].
* Parotids-SCRR (both parotids minus the SCRR), Submandibular glands (left and right combined) and Oral cavity are derived from the AI outputs.

* Note: Please contact the maintainers of [SMIT](github.com/cerr/smit) to request a link to the pre-trained weights (`weightsUrl`).

### References
1. Jiang, J., Tyagi, N., Tringale, K., Crane, C., & Veeraraghavan, H. Self-supervised 3D anatomy segmentation using self-distilled masked image transformer (SMIT). In International conference on medical image computing and computer-assisted intervention Sept. 2022; pp. 556-566. Cham: Springer Nature Switzerland.
2. van Rijn-Dekker, M.I.; van der Schaaf, A.; Nienhuis, S.W.; Arents-Huls, A.S.; Ger, R.B.; Hamming-Vrieze, O.; Hoebers, F.J.P.; de Ridder, M.; Vigorito, S.; Zwijnenburg, E.M.; et al. Clinical Introduction of Stem Cell Sparing Radiotherapy to Reduce the Risk of Xerostomia in Patients with Head and Neck Cancer. Cancers 2024, 16, 4283. https://doi.org/10.3390/cancers16244283
3. van Rijn-Dekker MI, van Luijk P, Schuit E, van der Schaaf A, Langendijk JA, Steenbakkers RJHM. Prediction of Radiation-Induced Parotid Gland-Related Xerostomia in Patients With Head and Neck Cancer: Regeneration-Weighted Dose. Int J Radiat Oncol Biol Phys. 2023 Nov 1;117(3):750-762. doi: 10.1016/j.ijrobp.2023.04.034. Epub 2023 May 6. PMID: 37150262.

## H&N DICOM dataset
This notebook assumes an input cohort of head-and-neck datasets (DICOM), one folder per patient, each holding the planning CT and RTDOSE:

```
dataDir/
    case_01/    (CT and RTDOSE DICOM files)
    case_02/
    ...
    case_10/
```

It also needs each patient's pre-treatment xerostomia (`pretreatment_toxicity`: 0 = No, 1 = Yes) in a CSV with one row per patient. `ID` must match the patient folder names. The sample file `sample_settings/clinical_predictors_xerostomia.csv` holds **placeholder values (all 0)**; that can be modified to match your dataset.

In [ ]:
import os
import yaml
import pandas as pd

### Data I/O

In [ ]:
dataDir = r'/path/to/hn_dicom/dataset'            # DICOM cohort: one folder per patient (CT + RTDOSE)
analysisDir = r'/path/to/logs_and_results'        # Directory for logs and results
installDir = r'/path/to/ai_model_installs'        # Location of the AI H&N CT segmentation model
settingsFile = r'sample_settings/ntcp_xerostomia_dicom.yaml'               # sample settings
predictorsCsv = r'sample_settings/clinical_predictors_xerostomia.csv'     # pretreatment_toxicity per patient
modelNum = 7                                      # CT_HN SMIT model
weightsUrl = None                                 # Link to the SMIT weights, from its maintainers

### Update the settings file to point to your data and model installation locations

In [ ]:
from validate.core import analysis

with open(settingsFile) as f:
    settings = yaml.safe_load(f)
settings['input']['data_dir'] = dataDir
settings['input']['predictors_csv'] = os.path.abspath(predictorsCsv)
settings['segment']['install_dir'] = installDir
analysis.saveSettings(settings, analysisDir)

### Run auto-segmentation

In [ ]:
from validate.seg.autoseg_pipeline import installMissingModels

# Install the selected model
installMissingModels([modelNum], installDir, progressCallback=print, weightsUrl=weightsUrl)

In [ ]:
from validate.run import applyAutoSegToCohort
from validate.logs import ErrorLog

# Run the SMIT model to segment the parotids, masseters, mandible, submandibular glands and oral cavity
errorLog = ErrorLog(analysisDir)
try:
    segResult = applyAutoSegToCohort(analysisDir, errorLog, stageCallback=print,
                                     progressCallback=lambda done, total, pid: print(f'{done}/{total}  {pid}'))
finally:
    errorLog.close()

### Compute NTCP
For each patient: derive the SCRR, Parotids-SCRR, Submandibular glands and Oral cavity from the AI outputs, compute the mean dose to each, and apply the xerostomia model with the patient's `pretreatment_toxicity`.

In [ ]:
from validate.run import applyModelToCohort

errorLog = ErrorLog(analysisDir)
try:
    ntcpResults = applyModelToCohort(analysisDir, errorLog,
                                     progressCallback=lambda done, total, pid: print(f'{done}/{total}  {pid}'))
finally:
    errorLog.close()

if (ntcpResults['status'] != 'ok').any():
    print(f'Some patients failed; see {errorLog.path}')
ntcpResults

### Save the NTCP results and list the derived structures

In [ ]:
from validate.logs import generateSummary

ntcpFile = os.path.join(analysisDir, 'ntcp_results.csv')
ntcpResults.to_csv(ntcpFile, index=False)

# Derived structures per case
summary = pd.read_csv(generateSummary(analysisDir))
summary[summary['ai_model'] == 'derived']

## Review sample results
Load a patient's CT and dose together with the derived structures, exported as RTSTRUCTs to the patient's folder in `analysisDir`.

In [ ]:
import glob
from cerr import plan_container as pc

sampleCase = r'case_01'
structNames = ['SCRR', 'Parotids-SCRR', 'Submandibular glands', 'Oral cavity']

derivedFiles = sorted(glob.glob(os.path.join(analysisDir, sampleCase, f'{sampleCase}_*_autoseg.dcm')))
planC = pc.loadDcmDir([os.path.join(dataDir, sampleCase)] + derivedFiles)
scanNum = 0
structNumV = [num for num, struct in enumerate(planC.structure) if struct.structureName in structNames]

In [ ]:
from cerr.viewer.pycerr_nbviewer import showNB
showNB(planC, scanNum=scanNum, struct_nums=structNumV, dose_nums=0,
       windowCenter=50, windowWidth=350)